**Yolo Training**

In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q ultralytics roboflow

import os
import yaml
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow
from ultralytics import YOLO

# 2. 이미 내려받은 데이터셋을 사용하고, 없으면 Roboflow에서 다운로드
# API 키는 환경 변수 또는 숨김 입력으로 전달합니다.
dataset_dir = Path("dataset")
data_yaml = dataset_dir / "data.yaml"
if not data_yaml.exists():
    api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("c-mk").project("yolo-tracking-oss1p")
    dataset = project.version(1).download("yolov11", location=str(dataset_dir), overwrite=True)
    data_yaml = Path(dataset.location) / "data.yaml"

# 실행 환경에 맞는 절대 경로를 설정 (Colab/Windows/Linux)
config = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
config["path"] = str(data_yaml.parent.resolve())
config.update(train="train/images", val="valid/images", test="test/images")
data_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

# 3. YOLOv11 Nano 파인튜닝
model = YOLO("yolo11n.pt")
results = model.train(
    data=str(data_yaml.resolve()),
    epochs=50,
    imgsz=640,
    batch=16,
)
